# Movie Recommendation System using LightFM

With LightFM, we would be able to approach the recommendation system with a hybrid approach of collaborative and cotent.

In [1]:
from lightfm import LightFM
from lightfm.datasets import fetch_movielens

# Use built-in sample dataset to test
data = fetch_movielens()
interactions = data['train']

model = LightFM(loss='warp')
model.fit(interactions, epochs=3)

In [ ]:
import pandas as pd 
# from sklearn.preprocessing import MultiLabelBinarizer
# from sklearn.metrics.pairwise import cosine_similarity
# from scipy.sparse import csr_matrix
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset
from scipy import sparse
# import os
# import boto3
# from dotenv import load_dotenv
import pickle


In [ ]:
load_dotenv()

bucket_name = os.getenv("AWS_BUCKET_NAME")
ratings_file = os.getenv("AWS_RATINGS_FILE")
models_file = os.getenv("AWS_MODEL_FILE")

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("AWS_SECRET"),
    region_name=os.getenv("AWS_REGION")
)

s3.download_file(bucket_name, ratings_file, "ratings.csv")
s3.download_file(bucket_name, models_file, models_file)

In [9]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv('../BigMovieData/ml-32m/movies.csv')

In [10]:
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))
all_genres = sorted(set(g for genre_list in movies['genres'] for g in genre_list))

In [11]:
dataset = Dataset()
dataset.fit(
    users=ratings['userId'].unique(),
    items=movies['movieId'].unique(),        # use all movies from movies.csv
    item_features=all_genres
)


In [12]:
(interactions, weights) = dataset.build_interactions([
    (row['userId'], row['movieId'], row['rating']) for _, row in ratings.iterrows()
])

In [ ]:
sparse.save_npz("interactions_matrix.npz", interactions)

# Save weights matrix
sparse.save_npz("weights_matrix.npz", weights)

In [13]:
movie_id_to_genres = {
    row['movieId']: row['genres'].split('|')
    for _, row in movies.iterrows()
    if isinstance(row['genres'], str) and row['genres'] != '(no genres listed)'
}

item_features = dataset.build_item_features(
    ((movie_id, genres) for movie_id, genres in movie_id_to_genres.items())
)

In [7]:
print(interactions.shape)

print(item_features.shape)

(610, 9742)
(9742, 9762)


In [14]:
model = LightFM(loss = "warp")
model.fit(interactions, item_features = item_features, epochs=3, num_threads=2)

In [15]:
with open("lightfm_model.pkl", "wb") as f:
    pickle.dump(f)

TypeError: dump() missing required argument 'file' (pos 2)

In [ ]:
with open("lightfm_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

In [24]:
personal_ratings = pd.read_csv("../personal_letterboxd/ratings.csv")
personal_ratings["Year"] = personal_ratings["Year"].astype(str)
personal_ratings['title'] = personal_ratings["Name"] + " (" + personal_ratings['Year'] + ")"
personal_ratings.head()

user_rating_merged = personal_ratings.merge(
    movies,
    left_on=["title"],
    right_on=["title"],
    how="inner"
)

user_rating_merged['userId'] = 200949
user_rating_merged.head()
final_user_rating = user_rating_merged[['userId', 'movieId', 'Rating']]
final_user_rating.rename(columns={"Rating": "rating"}, inplace=True)

C:\Users\larry\AppData\Local\Temp\ipykernel_16136\1813605711.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_user_rating.rename(columns={"Rating": "rating"}, inplace=True)


In [20]:
ratings = pd.concat([ratings, final_user_rating], ignore_index= True)

In [ ]:
interactions = sparse.load_npz("interactions_matrix.npz")
weights = sparse.load_npz("weights_matrix.npz")


In [ ]:
def recommend_movies_lightfm(user_id, movies_df, n_recommendations=10):
    n_users, n_items = interactions.shape
    
    # Get scores for all items for the given user
    scores = loaded_model.predict(user_id, np.arange(n_items))
    
    # Mask items already interacted with
    known_positives = interactions.tocsr()[user_id].indices
    scores[known_positives] = -np.inf  # Set already-watched movies to lowest score
    
    # Get top scores
    top_items = np.argsort(-scores)[:n_recommendations]
    
    # Get movie titles
    recommended_titles = movies_df[movies_df['movieId'].isin(top_items)][['movieId', 'title']]
    recommended_titles['score'] = recommended_titles['movieId'].apply(lambda x: scores[x])
    recommended_titles = recommended_titles.sort_values(by='score', ascending=False)
    
    return recommended_titles


('Planet Earth II (2016)', 4.455477720171993)
('The Work of Director Chris Cunningham (2003)', 4.417926060626628)
('I Am So Proud of You (2008)', 4.416756684529041)
('Twelve Angry Men (1954)', 4.4016952499585384)
('Planet Earth (2006)', 4.389601579900872)
('Cosmos', 4.382901230440704)
('Band of Brothers (2001)', 4.373974033992065)
('The Roosevelts: An Intimate History (2014)', 4.373105950965507)
('Dominion (2018)', 4.360389776641251)
('Shawshank Redemption, The (1994)', 4.339912271934893)
